In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib

sns.set_theme(style="white")

In [2]:
# Ruta de la carpeta donde se guardaron los datos
RUTA_DATOS_MODELO = Path("../datos_modelo")


# Cargamos las características
X_train = np.load(
    RUTA_DATOS_MODELO / "X_train.npy",
    allow_pickle=False
)

X_test = np.load(
    RUTA_DATOS_MODELO / "X_test.npy",
    allow_pickle=False
)


# Cargamos las etiquetas
y_train = np.load(
    RUTA_DATOS_MODELO / "y_train.npy",
    allow_pickle=False
)

y_test = np.load(
    RUTA_DATOS_MODELO / "y_test.npy",
    allow_pickle=False
)


# Cargamos los nombres de las características
nombres_caracteristicas = np.load(
    RUTA_DATOS_MODELO / "nombres_caracteristicas.npy",
    allow_pickle=True
)


# Mostramos las dimensiones para comprobar la carga
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print(
    "Nombres de características:",
    nombres_caracteristicas.shape
)

X_train: (12448, 120)
y_train: (12448,)
X_test: (3112, 120)
y_test: (3112,)
Nombres de características: (120,)


## 2. Entrenamiento del modelo Random Forest

En este paso se entrenará un modelo Random Forest utilizando las características extraídas de las ventanas de entrenamiento. Este algoritmo construye varios árboles de decisión y combina sus resultados para obtener una clasificación final.

El modelo recibirá `X_train`, que contiene las 120 características estadísticas de cada ventana, y `y_train`, que contiene la actividad correspondiente.

Se utilizarán 200 árboles y una semilla fija para que los resultados puedan reproducirse al ejecutar nuevamente el notebook. También se equilibrará la importancia de las clases debido a que algunas actividades contienen más muestras que otras.

El conjunto de prueba no participará durante el entrenamiento. Una vez ajustado el modelo, se utilizará `X_test` para generar las predicciones que posteriormente se compararán con `y_test`.

In [3]:
# ---------------------------------------------------------
# Creación del modelo Random Forest
# ---------------------------------------------------------

modelo_rf = RandomForestClassifier(
    n_estimators=200,       # Cantidad de árboles
    class_weight="balanced", # Compensa las diferencias entre actividades
    random_state=42,        # Permite reproducir los resultados
    n_jobs=-1               # Utiliza los núcleos disponibles
)


# ---------------------------------------------------------
# Entrenamiento del modelo
# ---------------------------------------------------------

modelo_rf.fit(
    X_train,
    y_train
)

print("Modelo Random Forest entrenado correctamente.")

Modelo Random Forest entrenado correctamente.


### Interpretación

El modelo Random Forest fue entrenado utilizando las ventanas del conjunto de entrenamiento. Cada ventana estuvo representada mediante 120 características estadísticas y su correspondiente etiqueta de actividad.

Durante este proceso, el modelo construyó 200 árboles de decisión. Cada árbol aprendió diferentes combinaciones y condiciones sobre las características, y sus resultados se combinarán mediante votación para clasificar nuevas ventanas.

El conjunto de prueba no se utilizó durante el entrenamiento, por lo que permanece reservado para evaluar el desempeño del modelo con datos que no fueron empleados para ajustarlo.

## 3. Predicción sobre el conjunto de prueba

Una vez entrenado el modelo, se utilizarán las características de las ventanas de prueba para predecir la actividad correspondiente a cada una.

Las predicciones generadas se guardarán en `y_pred`. Posteriormente se compararán con las etiquetas reales almacenadas en `y_test`.

In [4]:
# ---------------------------------------------------------
# Predicción de las actividades
# ---------------------------------------------------------

y_pred = modelo_rf.predict(X_test)


# Mostramos un resumen
print("Cantidad de predicciones:", len(y_pred))
print("Cantidad de etiquetas reales:", len(y_test))

print("\nPrimeras 10 predicciones:")
print(y_pred[:10])

print("\nPrimeras 10 etiquetas reales:")
print(y_test[:10])

Cantidad de predicciones: 3112
Cantidad de etiquetas reales: 3112

Primeras 10 predicciones:
['008' '009' '006' '006' '008' '008' '008' '008' '004' '004']

Primeras 10 etiquetas reales:
['009' '009' '009' '009' '008' '008' '008' '008' '004' '004']


### Interpretación

El modelo generó una predicción para cada ventana del conjunto de prueba. La cantidad de predicciones debe coincidir con la cantidad de etiquetas reales, lo que confirma que cada fila de `X_test` recibió una clasificación.

Las primeras predicciones permiten realizar una revisión inicial, pero todavía no son suficientes para determinar si el modelo funciona correctamente. Para evaluar su desempeño se deberán comparar todas las predicciones con las etiquetas reales mediante métricas de clasificación.

## 4. Evaluación del modelo Random Forest

Para evaluar el modelo se compararán las predicciones `y_pred` con las etiquetas reales `y_test`. Se calcularán exactitud, precisión, recall y F1-score.

Además, se construirá una matriz de confusión. En esta matriz, las filas representan las actividades reales y las columnas las actividades predichas por el modelo. Los valores de la diagonal corresponden a clasificaciones correctas, mientras que los valores fuera de la diagonal muestran las actividades que el modelo confundió.

In [6]:
# ---------------------------------------------------------
# Métricas generales del modelo
# ---------------------------------------------------------

exactitud = accuracy_score(
    y_test,
    y_pred
)

print(
    f"Exactitud general: {exactitud:.4f}"
)

print(
    f"Porcentaje de aciertos: {exactitud * 100:.2f}%"
)


# ---------------------------------------------------------
# Reporte por actividad
# ---------------------------------------------------------

print("\nReporte de clasificación:\n")

print(
    classification_report(
        y_test,
        y_pred,
        digits=4
    )
)

Exactitud general: 0.9569
Porcentaje de aciertos: 95.69%

Reporte de clasificación:

              precision    recall  f1-score   support

         000     1.0000    0.9931    0.9965       144
         001     0.8790    0.9079    0.8932       152
         002     0.8864    0.8603    0.8731       136
         003     0.9744    0.9500    0.9620       160
         004     0.9953    0.9727    0.9839       220
         005     0.9696    0.9955    0.9824       224
         006     0.9040    0.9524    0.9275       168
         007     0.9808    0.9968    0.9887       308
         008     0.9912    0.9417    0.9658       240
         009     0.9336    0.9795    0.9560       244
         010     0.9353    0.9400    0.9377       200
         011     0.9867    0.9737    0.9801       152
         012     0.9441    0.8438    0.8911       160
         013     0.9420    0.9591    0.9505       220
         014     0.9766    0.9676    0.9721       216
         015     0.9711    1.0000    0.9853       

### Interpretación de las métricas

El modelo Random Forest obtuvo una exactitud general de **95.69%**, lo que significa que clasificó correctamente aproximadamente 96 de cada 100 ventanas del conjunto de prueba.

El promedio macro del F1-score fue de **0.9529**. Este valor asigna la misma importancia a las 16 actividades y muestra que el buen resultado general no depende solamente de las clases con mayor cantidad de ventanas. El F1-score ponderado fue de **0.9568**, muy cercano a la exactitud general, lo que indica un desempeño relativamente consistente a pesar de las diferencias en el número de muestras por actividad.

Las actividades con mejor desempeño fueron:

- La actividad `000`, con un F1-score de **0.9965**.
- La actividad `007`, con un F1-score de **0.9887**.
- La actividad `015`, con un F1-score de **0.9853**.
- Las actividades `004`, `005` y `011`, con F1-scores superiores a **0.98**.

La actividad `015` obtuvo un recall de **1.0000**, por lo que todas sus ventanas reales fueron identificadas correctamente. Sin embargo, su precisión fue de **0.9711**, lo que indica que algunas ventanas de otras actividades fueron clasificadas incorrectamente como `015`.

Las actividades con mayor dificultad fueron:

- La actividad `002`, con un F1-score de **0.8731**.
- La actividad `001`, con un F1-score de **0.8932**.
- La actividad `012`, con un F1-score de **0.8911**.

La actividad `012` presentó el recall más bajo, con **0.8438**. Esto significa que aproximadamente el 84.38% de sus ventanas se reconocieron correctamente y que cerca del 15.62% se confundieron con otras actividades. En la matriz de confusión deberá revisarse principalmente hacia qué clases fueron dirigidos estos errores.

En general, los resultados muestran que las características estadísticas calculadas sobre ventanas de 220 puntos permiten diferenciar adecuadamente la mayoría de las actividades. No obstante, las actividades `001`, `002` y `012` requieren un análisis más detallado, ya que podrían presentar movimientos o patrones cinemáticos similares a los de otras clases.

## 5. Predicción por señal completa

El modelo genera una predicción para cada ventana de 220 puntos. Como cada señal original fue dividida en cuatro ventanas, se combinarán las cuatro predicciones para obtener una sola actividad por señal.

Para ello se utilizará el `sample_id` de los metadatos, que identifica la señal original de cada ventana. La actividad predicha con mayor frecuencia entre las cuatro ventanas se considerará la predicción final de la señal completa.

In [8]:
# ---------------------------------------------------------
# Carga de los metadatos de las ventanas de prueba
# ---------------------------------------------------------

metadatos_test = pd.read_csv(
    RUTA_DATOS_MODELO / "metadatos_test.csv",
    dtype={
        "sample_id": str,
        "actividad": str
    }
)


# Comprobamos que exista un metadato por cada predicción.
assert len(metadatos_test) == len(y_pred)


# ---------------------------------------------------------
# Tabla con la predicción de cada ventana
# ---------------------------------------------------------

resultados_ventanas = metadatos_test.copy()

resultados_ventanas["actividad_real"] = y_test
resultados_ventanas["actividad_predicha"] = y_pred


display(
    resultados_ventanas.head(8)
)

,sample_id,actividad,indice_original,numero_ventana,inicio,fin,actividad_real,actividad_predicha
0,009_0168,009,168,0,0,220,009,008
1,009_0168,009,168,1,220,440,009,009
2,009_0168,009,168,2,440,660,009,006
3,009_0168,009,168,3,660,880,009,006
4,008_0019,008,19,0,0,220,008,008
5,008_0019,008,19,1,220,440,008,008
6,008_0019,008,19,2,440,660,008,008
7,008_0019,008,19,3,660,880,008,008


In [9]:
# ---------------------------------------------------------
# Función para obtener la clase más frecuente
# ---------------------------------------------------------

def obtener_clase_mayoritaria(predicciones):
    """
    Devuelve la actividad predicha con mayor frecuencia
    entre las ventanas de una misma señal.
    """
    return predicciones.mode().iloc[0]


# ---------------------------------------------------------
# Una predicción final por señal completa
# ---------------------------------------------------------

resultados_senales = (
    resultados_ventanas
    .groupby("sample_id", as_index=False)
    .agg(
        actividad_real=(
            "actividad_real",
            "first"
        ),
        actividad_predicha=(
            "actividad_predicha",
            obtener_clase_mayoritaria
        ),
        cantidad_ventanas=(
            "numero_ventana",
            "count"
        )
    )
)


display(
    resultados_senales.head()
)

print(
    "Cantidad de señales evaluadas:",
    len(resultados_senales)
)

print(
    "Ventanas por señal:",
    resultados_senales["cantidad_ventanas"].unique()
)

,sample_id,actividad_real,actividad_predicha,cantidad_ventanas
0,000_0001,000,000,4
1,000_0014,000,000,4
2,000_0020,000,000,4
3,000_0021,000,000,4
4,000_0037,000,000,4


Cantidad de señales evaluadas: 778
Ventanas por señal: [4]


In [10]:
# ---------------------------------------------------------
# Etiquetas reales y predicciones finales
# ---------------------------------------------------------

y_test_senales = resultados_senales[
    "actividad_real"
].to_numpy()

y_pred_senales = resultados_senales[
    "actividad_predicha"
].to_numpy()


# ---------------------------------------------------------
# Exactitud por señal completa
# ---------------------------------------------------------

exactitud_senales = accuracy_score(
    y_test_senales,
    y_pred_senales
)

print(
    f"Exactitud por señal completa: "
    f"{exactitud_senales:.4f}"
)

print(
    f"Porcentaje de aciertos por señal: "
    f"{exactitud_senales * 100:.2f}%"
)

print("\nReporte por señal completa:\n")

print(
    classification_report(
        y_test_senales,
        y_pred_senales,
        digits=4
    )
)

Exactitud por señal completa: 0.9756
Porcentaje de aciertos por señal: 97.56%

Reporte por señal completa:

              precision    recall  f1-score   support

         000     1.0000    1.0000    1.0000        36
         001     0.8780    0.9474    0.9114        38
         002     0.9118    0.9118    0.9118        34
         003     1.0000    1.0000    1.0000        40
         004     1.0000    1.0000    1.0000        55
         005     1.0000    1.0000    1.0000        56
         006     0.8913    0.9762    0.9318        42
         007     1.0000    1.0000    1.0000        77
         008     1.0000    0.9833    0.9916        60
         009     0.9677    0.9836    0.9756        61
         010     1.0000    0.9400    0.9691        50
         011     1.0000    1.0000    1.0000        38
         012     0.9444    0.8500    0.8947        40
         013     0.9818    0.9818    0.9818        55
         014     1.0000    0.9815    0.9907        54
         015     0.9767    

### Interpretación de la evaluación por señal completa

Después de combinar mediante votación mayoritaria las cuatro predicciones correspondientes a cada señal, el modelo obtuvo una exactitud de **97.56%** sobre las 778 señales del conjunto de prueba. Esto significa que clasificó correctamente aproximadamente 98 de cada 100 señales completas.

Este resultado es superior a la exactitud de **95.69%** obtenida por ventana. La mejora indica que algunas ventanas individuales fueron clasificadas incorrectamente, pero las demás ventanas de la misma señal permitieron recuperar la actividad correcta mediante la votación mayoritaria.

El F1-score macro fue de **0.9717**, lo que indica que el desempeño fue alto al asignar la misma importancia a cada una de las 16 actividades. El F1-score ponderado fue de **0.9756**, prácticamente igual a la exactitud general. La cercanía entre estas métricas muestra que el buen resultado no se explica únicamente por las actividades con mayor cantidad de señales.

Las actividades `000`, `003`, `004`, `005`, `007` y `011` alcanzaron una clasificación perfecta en este conjunto de prueba, con precision, recall y F1-score iguales a **1.0000**.

La actividad con mayor dificultad continuó siendo la `012`, con un recall de **0.8500** y un F1-score de **0.8947**. Esto significa que el modelo reconoció correctamente 85% de sus señales, mientras que el 15% restante fue confundido con otras actividades.

Las actividades `001` y `002` también presentaron resultados inferiores al promedio, con F1-scores de **0.9114** y **0.9118**, respectivamente. En el caso de la actividad `006`, el recall fue alto, con **0.9762**, pero su precisión fue de **0.8913**. Esto sugiere que el modelo recuperó casi todas las señales reales de `006`, aunque también clasificó incorrectamente señales de otras actividades dentro de esta clase.

En general, la combinación de las predicciones de las cuatro ventanas permitió obtener una clasificación más estable de la señal completa. La matriz de confusión por señal deberá utilizarse para identificar específicamente con qué actividades se confundieron las clases `001`, `002`, `006` y `012`.